# Train a model on the data

In [ ]:

from collections import Counter
import jsonlines
import pandas as pd
import numpy as np
from numpy.typing import ArrayLike
import os

from ignite.engine import Engine, Events
from ignite.handlers import ProgressBar
from ignite.metrics import Recall, Precision, Accuracy, Loss

import torch
from torch import nn
import torch.nn.functional as F
import torch_geometric
from torch_geometric.data import HeteroData
from torch_geometric.nn import GraphConv, SAGEConv, to_hetero, HeteroConv
from torch_geometric import transforms as T
from torch_geometric import seed_everything

from jazz_graph.data.reporting import inspect_degrees
from jazz_graph.training.logging import (
    ExperimentLogger,
    save_embeddings_handler,
    save_checkpoint_handler,
    run_evaluator_handler,
    log_experiment_handler,
    console_logging,
    binary_output_transform
)
from jazz_graph.data.graph_builder import CreateTensors, prune_isolated_nodes, make_jazz_data
from jazz_graph.model.model import JazzModel, LinkPredictionModel, NodeClassifier


In [ ]:
models_dir = '/workspace/local_data/graph_parquet_proto'
assert os.path.exists(models_dir)
create = CreateTensors(models_dir)

In [ ]:
data = make_jazz_data(create)

In [ ]:
# TODO: report on the data a little more concreately.
# E.g., who are the hub nodes? How many nodes have > 50 edges.
# how many nodes have < 6 edges? All these, by type.
# Get really fancy and visualize a sub-graph.

def frequency_of_n_labels(data: HeteroData):
    """Return frequency of number of labels in the data, i.e., what percentage have 1 label, 0 labels, etc."""
    count_by_row = data['performance'].y.sum(dim=1)
    n_samples = data['performance'].y.shape[0]
    counter = Counter((int(x) for x in (count_by_row)))
    for i in range(len(counter)):
        count = counter[i]
        freq = count / n_samples
        print(f"Num samples with {i} labels: {freq:.3f}")

In [ ]:
data.metadata()

In [ ]:
create.label_names()

In [ ]:
print(data)
print(
    f"The graph contains {'' if data.has_isolated_nodes() else 'no '}isolated nodes and",
    f"is {'directed' if data.is_directed() else 'undirected'}."
)
frequency_of_n_labels(data)
for style, count in (zip(create.label_names(), data['performance'].y.sum(dim=0))):
    print(f"  {style}: {int(count) / create._labels.shape[0]:.1%}")
    # Easy Listening is probably a mislabel by modern standards.


In [ ]:
inspect_degrees(data)

## Model

## Train Style Classifier

In [ ]:
from torch_geometric.loader import NeighborLoader

def train_indicies(mask):
    num_nodes = mask.shape[0]
    all_node_indicies = torch.arange(num_nodes)
    return all_node_indicies[mask]


data_config = {
    'dataset': models_dir,
    'sampling': {'num_neighbors': [5, 5, 3]}
}

sampling = data_config['sampling']['num_neighbors']
train_loader = NeighborLoader(
    data,
    sampling,
    batch_size=128,
    input_nodes=('performance', train_indicies(data['performance'].train_mask)),
    shuffle=True
)
dev_loader = NeighborLoader(
    data,
    sampling,
    batch_size=128,
    input_nodes=('performance', train_indicies(data['performance'].dev_mask)),
    shuffle=True
)

### Train function

In [ ]:
class GNNTrainingLogic:
    """Define training step and eval steps."""
    def __init__(self, model, optimizer, criterion):
        self.device = next(model.parameters()).device
        self.model = model
        self.optimizer = optimizer
        self.criterion = criterion

    def train_step(self, engine, batch: HeteroData) -> dict:
        """Complete one step of gradient descent."""
        self.model.train()
        self.optimizer.zero_grad()
        batch.to(self.device)
        batch_size = batch['performance'].batch_size
        y_pred = model(batch)[:batch_size]
        y_true = batch['performance'].y[:batch_size].to(torch.float)
        loss = self.criterion(y_pred, y_true.to(torch.float))  # TODO: need to cast here is just an artifact of wrong type in data.
        loss.backward()
        self.optimizer.step()
        return {'loss': loss.item(), 'y_pred': y_pred.detach(), 'y_true': y_true.detach()}

    def eval_step(self, engine, batch: HeteroData) -> dict:
        """Complete one pass over a batch of data with no-grad and return results."""
        self.model.eval()
        batch.to(self.device)

        batch_size = batch['performance'].batch_size
        with torch.no_grad():
            y_pred = model(batch)[:batch_size]
            y_true = batch['performance'].y[:batch_size].to(torch.float)
            loss = self.criterion(y_pred, y_true.to(torch.float))
        return {'y_pred': y_pred, 'y_true': y_true}

model_config = {
    'hidden_dim': 128,
    'embed_dim': 64,
    # NOTE: sage seems to need high regularization, >.4
    #       gat does better with less, ~=.2
    'dropout': 0.2,
    'model_type': 'gat'
}

model = NodeClassifier(
    JazzModel(
        data['performance'].num_nodes,
        data['artist'].num_nodes,
        data['song'].num_nodes,
        hidden_dim=model_config['hidden_dim'],
        embed_dim=model_config['embed_dim'],
        dropout=model_config['dropout'],
        metadata=data.metadata(),
        model_type=model_config['model_type']
    ),
    hidden_dim=model_config['hidden_dim'],
    num_classes=len(create.label_names())
)


experiment_config = {
    'data_config': data_config,
    'model': model_config,
    'dataset': models_dir,
    'lr': .008,
    'batch_size': train_loader.batch_size
}


criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=experiment_config['lr'])
trainer_logic = GNNTrainingLogic(model, optimizer, criterion)
# assert trainer_logic.train_step(None, batch) is not None, "This is really just checking that the forward pass works."

experiment_logger = ExperimentLogger(root='../experiments', run_name=f'gnn_classifier_{os.path.basename(models_dir)}', config=experiment_config)

trainer = Engine(trainer_logic.train_step)
# train_evaluator = Engine(trainer_logic.eval_step)  # Wastful! Takes extra time!
dev_evaluator = Engine(trainer_logic.eval_step)

metrics = {
    'accuracy': Accuracy(output_transform=binary_output_transform),
    'recall': Recall(output_transform=binary_output_transform),
    'precision': Precision(output_transform=binary_output_transform),
    'loss': Loss(criterion, output_transform=lambda out: (out['y_pred'], out['y_true']))
}

for name, metric in metrics.items():
    metric.attach(trainer, name)
    metric.attach(dev_evaluator, name)

progress_bar = ProgressBar()
progress_bar.attach(trainer, metric_names=['loss'])

# trainer.add_event_handler(Events.EPOCH_COMPLETED, run_evaluator_handler, train_evaluator, train_loader, "Training")
trainer.add_event_handler(Events.EPOCH_COMPLETED, console_logging, 'Training', trainer)
trainer.add_event_handler(Events.EPOCH_COMPLETED, run_evaluator_handler, dev_evaluator, dev_loader)
trainer.add_event_handler(Events.EPOCH_COMPLETED, log_experiment_handler, experiment_logger, 'train', trainer)
dev_evaluator.add_event_handler(Events.EPOCH_COMPLETED, log_experiment_handler, experiment_logger, 'dev', trainer)
dev_evaluator.add_event_handler(Events.EPOCH_COMPLETED, console_logging, 'Valiation', trainer)
trainer.add_event_handler(Events.COMPLETED, save_embeddings_handler, experiment_logger, model)
trainer.add_event_handler(Events.COMPLETED, experiment_logger.save_checkpoint)


In [ ]:
trainer.run(train_loader, max_epochs=30)

In [ ]:
from jazz_graph.training.logging import plot_logs


plot_logs(experiment_logger.run_dir)

In [ ]:
batch

In [ ]:
torch.tensor([1]).tolist()

In [ ]:
batch = next(iter(dev_loader))
from jazz_graph.data.reporting import visualize_hetero_graph
visualize_hetero_graph(batch)